In [13]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import zipfile
import os
import urllib.request
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial import distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [93]:
class brain(nn.Module):
    def __init__(self, hidden_wake_size, hidden_sleep_size, sleep_output_size, vocab_size, embedding_dim=32, num_layers=1, num_layers_sleep=1):
        super(brain, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(hidden_wake_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        # self.fc = nn.Linear(hidden_wake_size, 15)
        self.wake_fc = nn.Linear(hidden_wake_size, vocab_size)
        self.sleep_output_size = sleep_output_size

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False):
        # print(x.shape, 'x')
        x = self.embedding(x)
        
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.sleep_output_size))
            
        # print(x.size())
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        # out_ = self.fc(out) 
        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw


In [9]:
raw_text[:1000]

' anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term is still used in a pejorative way to describe any act that used violent means to destroy the organization of society it has also been taken up as a positive label by self defined anarchists the word anarchism is derived from the greek without archons ruler chief king anarchism as a political philosophy is the belief that rulers are unnecessary and should be abolished although there are differing interpretations of what this means anarchism also refers to related social movements that advocate the elimination of authoritarian institutions particularly the state the word anarchy as most anarchists use it does not imply chaos nihilism or anomie but rather a harmonious anti authoritarian society in place of what are regarded as authoritarian political structures and coercive economic institut

In [90]:
# Step 1: Download and extract text8
def download_text8(path="dataset/text8.zip"):
    url = "http://mattmahoney.net/dc/text8.zip"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("Downloading text8...")
        urllib.request.urlretrieve(url, path)
    with zipfile.ZipFile(path) as zf:
        data = zf.read(zf.namelist()[0]).decode('utf-8')
    return data

# Step 2: Build character-level vocabulary
def build_vocab(text):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

# Step 3: Encode text into integer tokens
def encode(text, stoi):
    return np.array([stoi[c] for c in text], dtype=np.int32)

# Step 4: Create input and target sequences
#def create_dataset(encoded_text, seq_len=64, step=1):
#    inputs, targets = [], []
#    for i in range(0, len(encoded_text) - seq_len, step):
#        inputs.append(encoded_text[i:i+seq_len])
#        targets.append(encoded_text[i+1:i+seq_len+1])
#    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

class Dataset_converter(Dataset):
    def __init__(self, encoded_text, working_memory=1, short_term_memory=8):
        
        self.X = []
        self.y = []

        for ii in range(0, len(encoded_text) - working_memory - short_term_memory, 1):
            self.X.append(
                encoded_text[ii:ii+short_term_memory]
            )
            self.y.append(
                encoded_text[ii+short_term_memory]
            )

        self.X = tnsr(np.array(self.X)).long()
        self.y = tnsr(np.array(self.y)).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [91]:
text = download_text8()
stoi, itos = build_vocab(text)
encoded = encode(text[:100000], stoi)

data_set = Dataset_converter(encoded, 1, 3)

In [168]:
hidden_wake_size = 512
hidden_sleep_size = 10
sleep_output_size = 5
working_memory = 1
short_term_memory = 30

text = download_text8()
stoi, itos = build_vocab(text)
encoded = encode(text[:2000], stoi)  # truncate for faster training
data_set = Dataset_converter(encoded, working_memory=working_memory, short_term_memory=short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

vocab_size = len(stoi)
model = brain(hidden_wake_size, hidden_sleep_size, sleep_output_size, vocab_size, embedding_dim=256)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

batch_size = 1
total = 0
correct = np.zeros(1000,dtype=float)

for _ in range(50):
    for x, y in train_loader:
    
        optimizer.zero_grad()
    
        if i == 0:
            logits, hidden = model(x)
        else:
            logits, hidden = model(x, hidden)
            
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
    
        with torch.no_grad():
            estimated_y = logits.argmax(axis=1)
    
            total += 1
            if y == estimated_y:
                    correct[total%1000] = 1
            else:
                correct[total%1000] = 0
    
            if total%1000 == 0:
                print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {np.sum(correct)/100:.4f}')

Iter : 1001, loss: 2.7811, accuracy: 2.4800
Iter : 2001, loss: 3.3096, accuracy: 2.8500
Iter : 3001, loss: 5.0354, accuracy: 3.2300
Iter : 4001, loss: 3.2158, accuracy: 3.4400
Iter : 5001, loss: 2.4035, accuracy: 3.9900
Iter : 6001, loss: 0.9205, accuracy: 3.9700
Iter : 7001, loss: 3.3194, accuracy: 4.3700
Iter : 8001, loss: 2.6612, accuracy: 4.4500
Iter : 9001, loss: 0.2436, accuracy: 5.0300
Iter : 10001, loss: 3.9381, accuracy: 5.2400
Iter : 11001, loss: 1.3620, accuracy: 5.1900
Iter : 12001, loss: 0.6576, accuracy: 5.4500
Iter : 13001, loss: 0.3532, accuracy: 5.3300
Iter : 14001, loss: 2.6908, accuracy: 5.5300
Iter : 15001, loss: 1.7365, accuracy: 5.9400
Iter : 16001, loss: 0.0000, accuracy: 5.8600
Iter : 17001, loss: 0.0088, accuracy: 6.0100
Iter : 18001, loss: 0.0046, accuracy: 6.3700
Iter : 19001, loss: 0.3844, accuracy: 6.4700
Iter : 20001, loss: 0.7934, accuracy: 6.4600
Iter : 21001, loss: 0.0002, accuracy: 6.3800
Iter : 22001, loss: 0.3372, accuracy: 6.4100
Iter : 23001, loss:

KeyboardInterrupt: 

In [169]:
total_replay_sample = 10000

X_hat = torch.randint(0, vocab_size, (1,)) 
seq = ''

for jj in range(total_replay_sample):
    with torch.no_grad():
        # print(tokens[idx])
        if jj == 0:
            X_hat_, hidden_state = model(X_hat.reshape(1,-1))
        else:
            X_hat_, hidden_state = model(X_hat.reshape(1,-1), hidden_state)

            
        X_hat = torch.nn.functional.softmax(X_hat_, dim=1)
        
        dist_categ = torch.distributions.Categorical(probs=X_hat.reshape(-1))
        X_hat = dist_categ.sample()

        seq = seq + itos[int(X_hat)]
        

In [170]:
text[:2000]

' anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term is still used in a pejorative way to describe any act that used violent means to destroy the organization of society it has also been taken up as a positive label by self defined anarchists the word anarchism is derived from the greek without archons ruler chief king anarchism as a political philosophy is the belief that rulers are unnecessary and should be abolished although there are differing interpretations of what this means anarchism also refers to related social movements that advocate the elimination of authoritarian institutions particularly the state the word anarchy as most anarchists use it does not imply chaos nihilism or anomie but rather a harmonious anti authoritarian society in place of what are regarded as authoritarian political structures and coercive economic institut

In [171]:
seq

'e tis d gan tis tutite titis tite gatis titede tit te te titis gatite titis te tis tis titis itititingatis tince itis tis tis tis s de tititititis gatis itis d tis gare s gan gatitis tititis tin titis te tititis s outis gare tis te tioute tin tis tis tis tititititis s te titititis is atitithons d titis tis te tis tititingatititititis s gare itis tincis te gatis garoupon titis tititis te tis te titis gatitititis oum titinatis tititis tious tite titititis atis gatitite tis ga ite tis ore titititis ga tis tis tis tititititititis gatitis tititingatis tisoutis atis tititis s titis on gare gatis gatitis te te titis ilan ga oure in gatitis tite titis tititititis tis tis d titite te tis tititin tititincore te tite tinare gare tititis te ous te titis s titite tititis te titis ous tititititis tis titin gare tis titis titititis titititite tis garoutis gate d utitititititilare tis tis gat tis tie gate tis tis tinare tins be is tis tis titititis be ongare titis tititititis titititis gatititis gan 

In [70]:
bptt_steps = 8  # Number of time steps to backpropagate through
batch_size = 1
model.train()
hidden = None

total = 0
correct = np.zeros(1000, dtype=float)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

for i in range(0, len(inputs) - batch_size - bptt_steps, bptt_steps):

    # Accumulate loss over `bptt_steps`
    loss_total = 0.0
    optimizer.zero_grad()

    for j in range(bptt_steps):
        idx = i + j

        x = inputs[idx:idx+batch_size]
        y = targets[idx:idx+batch_size]

        if hidden is not None:
            # Detach hidden state from previous graph
            hidden = hidden.detach()

        logits, hidden = model(x, hidden)

        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss_total += loss

        with torch.no_grad():
            estimated_y = logits.argmax(dim=1)
            total += 1
            correct[total % 1000] = (estimated_y == y.view(-1)).float().mean().item()

    # Backward through the accumulated loss
    loss_total.backward()
    optimizer.step()

    if total % 1000 == 0:
        print(f"Iter: {total}, Avg Loss: {loss_total.item()/bptt_steps:.4f}, Accuracy: {np.sum(correct)/10:.2f}%")


ValueError: Expected input batch_size (1) to match target batch_size (8).

In [71]:
x.shape

torch.Size([1, 8])

In [80]:
targets[0]

tensor([ 1, 14,  1, 18,  3,  8,  9, 19])

In [77]:
targets[1]

tensor([14,  1, 18,  3,  8,  9, 19, 13])

In [78]:
x[0]

tensor([ 0,  1, 14,  1, 18,  3,  8,  9])

In [81]:
inputs[0]

tensor([ 0,  1, 14,  1, 18,  3,  8,  9])

In [ ]:
us, an, ed